# Delta Decomposition Analysis

**Goal:** mathematically prove the "Master/Slave" dynamic by decomposing the value incentives.

**Methodology:**
We run a simulation. Whenever an agent is 1 step away from an apple (an **Opportunity**), we calculate:
1.  **V(Pick):** The values ($V_0, V_1$) associated with the best move that *lands on* the apple.
2.  **V(Avoid):** The values ($V_0, V_1$) associated with the best move that *does not* land on the apple.
3.  **Delta ($\Delta$):** $V(Pick) - V(Avoid)$.

**Hypothesis to Validate:**
1.  **Agent 0 (The Master):** $\Delta V_{self} < 0$ (Laziness) is stronger than $\Delta V_{other}$ (Peer Pressure). Result: Avoids.
2.  **Agent 1 (The Slave):** $\Delta V_{self} < 0$ (Fear) is overpowered by $\Delta V_{other} \gg 0$ (Master's Demand). Result: Picks.

In [5]:
# === CONFIGURATION ===
HEIGHT = 6
WIDTH = 6
NUM_AGENTS = 2
DISCOUNT = 0.99
SPAWN_PROB = 0.05
DESPAWN_PROB = 0.09
CNN_CONV_CHANNELS = [32, 64]
CNN_HEAD_HIDDEN_DIM = 128
CNN_HEAD_NUM_LAYERS = 3
CNN_KERNEL_SIZE = 3

SIMULATION_STEPS = 2000

PATH_DECENTRALIZED_AG0 = "decen_cnn_-12_reward/model_agent0_step16050000.pt"
PATH_DECENTRALIZED_AG1 = "decen_cnn_-12_reward/model_agent1_step16050000.pt"

In [6]:
import torch
import numpy as np
from tqdm import tqdm
import sys
sys.path.append("../") 
from models.value_cnn_new import ValueCNNDecentralizedStandard
from tadd_helpers.env_functions import init_empty_state, spawn_apples, despawn_apples, State
from orchard.environment import MoveAction

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [7]:
# Load Models
dec_models = []
for p in [PATH_DECENTRALIZED_AG0, PATH_DECENTRALIZED_AG1]:
    m = ValueCNNDecentralizedStandard(HEIGHT, WIDTH, 0.0, DISCOUNT, CNN_HEAD_HIDDEN_DIM, CNN_HEAD_NUM_LAYERS, CNN_CONV_CHANNELS, CNN_KERNEL_SIZE).to(DEVICE)
    m.load_state_dict(torch.load(p, map_location=DEVICE))
    m.eval()
    dec_models.append(m)

print("Models loaded.")

Models loaded.


In [8]:
def get_vals(state):
    v0 = dec_models[0].get_value(state, agent_pos=state.agent_position(0))
    v1 = dec_models[1].get_value(state, agent_pos=state.agent_position(1))
    return v0, v1, v0+v1

def run_delta_decomposition():
    state = init_empty_state(HEIGHT, WIDTH, NUM_AGENTS)
    
    # Store Delta Data: {'delta_self': float, 'delta_other': float, 'picked': bool}
    data = {
        0: [], # When Agent 0 Acts
        1: []  # When Agent 1 Acts
    }
    
    print(f"Running Delta Decomposition for {SIMULATION_STEPS} steps...")
    
    for _ in tqdm(range(SIMULATION_STEPS)):
        active_idx = state.get_random_agent_id()
        curr_pos = state.agent_position(active_idx)
        
        # 1. Analyze All Possible Moves
        possible_moves = []
        
        for action in MoveAction:
            # Physics
            new_pos = np.clip(curr_pos + action.vector, [0, 0], [HEIGHT-1, WIDTH-1])
            s_mid = state.copy()
            s_mid.set_agent_position(active_idx, new_pos)
            
            # Outcome Check
            lands_on_apple = (state.apples[tuple(new_pos)] > 0)
            
            # Value Query (Policy uses Sum of V_mid)
            v0, v1, vsum = get_vals(s_mid)
            
            possible_moves.append({
                'action': action,
                'lands_on_apple': lands_on_apple,
                'v0': v0,
                'v1': v1,
                'vsum': vsum
            })
            
        # 2. Identify Opportunity
        # Must have at least one move that picks and one move that avoids
        pick_options = [m for m in possible_moves if m['lands_on_apple']]
        avoid_options = [m for m in possible_moves if not m['lands_on_apple']]
        
        if pick_options and avoid_options:
            # Find Best Pick and Best Avoid (according to Team Sum)
            best_pick = max(pick_options, key=lambda x: x['vsum'])
            best_avoid = max(avoid_options, key=lambda x: x['vsum'])
            
            # Calculate Deltas (Pick - Avoid)
            # If Delta > 0, Picking is preferred.
            delta_v0 = best_pick['v0'] - best_avoid['v0']
            delta_v1 = best_pick['v1'] - best_avoid['v1']
            
            # Determine if we actually picked
            # The policy takes the absolute max of ALL moves
            all_best = max(possible_moves, key=lambda x: x['vsum'])
            did_pick = all_best['lands_on_apple']
            
            # Store Data relative to SELF and OTHER
            other_idx = 1 - active_idx
            
            if active_idx == 0:
                d_self = delta_v0
                d_other = delta_v1
            else:
                d_self = delta_v1
                d_other = delta_v0
                
            data[active_idx].append({
                'delta_self': d_self,
                'delta_other': d_other,
                'picked': did_pick
            })
            
            # Choose action for simulation
            chosen_action = all_best['action']
        else:
            # No opportunity, just churn
            # If avoiding, picking best avoid. If no avoid (trapped?), pick random.
            if possible_moves:
                chosen_action = max(possible_moves, key=lambda x: x['vsum'])['action']
            else:
                chosen_action = MoveAction.STAY

        # 3. Execute Step
        new_pos = np.clip(curr_pos + chosen_action.vector, [0,0], [HEIGHT-1, WIDTH-1])
        state.set_agent_position(active_idx, new_pos)
        if state.apples[tuple(new_pos)] > 0: state.remove_apple_at(new_pos)
        spawn_apples(state, SPAWN_PROB)
        despawn_apples(state, DESPAWN_PROB)
        
    return data

results = run_delta_decomposition()

Running Delta Decomposition for 2000 steps...


100%|██████████| 2000/2000 [00:52<00:00, 38.15it/s] 


In [9]:
# === REPORT GENERATION ===

def print_analysis(agent_id, logs):
    if not logs: 
        print(f"No opportunities found for Agent {agent_id}")
        return

    count = len(logs)
    picks = sum(1 for x in logs if x['picked'])
    pick_rate = picks / count * 100
    
    # Calculate Means
    avg_delta_self = np.mean([x['delta_self'] for x in logs])
    avg_delta_other = np.mean([x['delta_other'] for x in logs])
    avg_delta_team = avg_delta_self + avg_delta_other
    
    print(f"\n{'='*40}")
    print(f"AGENT {agent_id} ANALYSIS (N={count} Opportunities)")
    print(f"{'='*40}")
    print(f"PICK RATE: {pick_rate:.1f}%")
    
    print(f"\nINCENTIVE BREAKDOWN (Avg Delta V = Pick - Avoid):")
    print(f"  Self Incentive (Delta V_self):   {avg_delta_self:>6.2f}")
    print(f"  Peer Incentive (Delta V_other):  {avg_delta_other:>6.2f}")
    print(f"  ---------------------------------------")
    print(f"  NET TEAM INCENTIVE:              {avg_delta_team:>6.2f}")
    
    print(f"\nINTERPRETATION:")
    if avg_delta_self < 0:
        print(f"  - Agent {agent_id} prefers AVOIDING (Self-Interest is Negative).")
    else:
        print(f"  - Agent {agent_id} prefers PICKING (Self-Interest is Positive).")
        
    if avg_delta_other > 0:
        print(f"  - The Partner wants Agent {agent_id} to PICK.")
    else:
        print(f"  - The Partner wants Agent {agent_id} to AVOID.")
        
    if avg_delta_team > 0:
        print(f"  => RESULT: Team Policy generally forces a PICK.")
    else:
        print(f"  => RESULT: Team Policy generally forces an AVOID.")

print_analysis(0, results[0])
print_analysis(1, results[1])


AGENT 0 ANALYSIS (N=617 Opportunities)
PICK RATE: 38.1%

INCENTIVE BREAKDOWN (Avg Delta V = Pick - Avoid):
  Self Incentive (Delta V_self):    -0.85
  Peer Incentive (Delta V_other):    0.71
  ---------------------------------------
  NET TEAM INCENTIVE:               -0.14

INTERPRETATION:
  - Agent 0 prefers AVOIDING (Self-Interest is Negative).
  - The Partner wants Agent 0 to PICK.
  => RESULT: Team Policy generally forces an AVOID.

AGENT 1 ANALYSIS (N=674 Opportunities)
PICK RATE: 86.5%

INCENTIVE BREAKDOWN (Avg Delta V = Pick - Avoid):
  Self Incentive (Delta V_self):    -0.48
  Peer Incentive (Delta V_other):    0.93
  ---------------------------------------
  NET TEAM INCENTIVE:                0.44

INTERPRETATION:
  - Agent 1 prefers AVOIDING (Self-Interest is Negative).
  - The Partner wants Agent 1 to PICK.
  => RESULT: Team Policy generally forces a PICK.
